# AI Journal Club App: Colab Notebook Prototype

This notebook builds the core AI ideas for James's AI Journal Club App:

1. A high-school friendly library of frontier AI videos and tutorials.
2. A supervised machine learning recommender based on club survey preferences and likes.
3. A small neural-network concept demo for interest prediction.
4. A reinforcement learning feedback loop that updates recommendations from likes/dislikes.
5. Data exports that can be used by the Streamlit app in this package.

The notebook is designed for free Google Colab and does not require paid APIs.

In [ ]:
# Colab setup. Run this cell first.
!pip -q install pandas numpy scikit-learn matplotlib

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

print("Python:", sys.version.split()[0])

## 1. Create sample club data

Replace these sample rows with James's real journal club content later. The current rows are enough to demonstrate the AI architecture.

In [ ]:
from io import StringIO

videos_csv = 'content_id,title,topic,level,summary,tags,source_url,estimated_minutes,format,freshness_days,frontier_theme\nV001,AI Agents: Digital Helpers That Plan,AI Agents,Beginner,"An introduction to agentic AI systems that can plan, call tools, and check their own work.",agents;planning;tools;automation,https://www.youtube.com/results?search_query=AI+agents+explained+for+students,12,Video,7,Agentic workflows\nV002,Transformers and Attention with a Classroom Analogy,Deep Learning,Beginner,"Explains how attention lets a model focus on the most important words or image regions.",transformers;attention;language+models,https://www.youtube.com/results?search_query=transformers+attention+explained+high+school,15,Video,14,Foundation models\nV003,Reinforcement Learning: How AI Learns from Feedback,Reinforcement Learning,Beginner,"Shows how rewards, actions, and policies help AI improve through trial and feedback.",reinforcement+learning;feedback;reward,https://www.youtube.com/results?search_query=reinforcement+learning+explained+for+beginners,14,Video,21,Self-improving systems\nV004,AI Safety and Alignment Basics,AI Safety,Beginner,"Introduces why AI systems need guardrails, evaluation, transparency, and human oversight.",safety;alignment;ethics;evaluation,https://www.youtube.com/results?search_query=AI+safety+alignment+basics+students,13,Video,10,Responsible AI\nV005,Multimodal AI: Models That See Hear and Read,Multimodal AI,Beginner,"Explains models that combine text, images, audio, and video to understand the world.",multimodal;vision;audio;text,https://www.youtube.com/results?search_query=multimodal+AI+explained+for+students,11,Video,6,Multimodal foundation models\nV006,TinyML and Edge AI on Small Devices,Deployment,Intermediate,"Describes how AI can run on phones, sensors, and low-power devices.",tinyml;edge+ai;deployment;devices,https://www.youtube.com/results?search_query=TinyML+edge+AI+explained,16,Video,30,Efficient AI\nV007,RAG: Teaching an AI to Use a Library,Retrieval-Augmented Generation,Beginner,"Explains retrieval-augmented generation using a library and note-card analogy.",RAG;retrieval;knowledge;base;LLM,https://www.youtube.com/results?search_query=RAG+retrieval+augmented+generation+explained,12,Video,9,Knowledge-grounded AI\nV008,Diffusion Models: How AI Draws Images Step by Step,Generative AI,Beginner,"Introduces image generation by gradually removing noise until a picture appears.",diffusion;image+generation;generative+AI,https://www.youtube.com/results?search_query=diffusion+models+explained+for+beginners,10,Video,18,Generative media\nV009,Prompt Engineering and AI Evaluation,Prompting,Beginner,"Shows how prompts guide AI systems and why outputs should be tested carefully.",prompting;evaluation;benchmarks;LLM,https://www.youtube.com/results?search_query=prompt+engineering+AI+evaluation+students,9,Tutorial,12,Human-AI interaction\nV010,Robotics Plus AI: From Simulation to the Real World,Robotics,Intermediate,"Connects robotics, simulation, sensors, and AI decision making.",robotics;simulation;control;sensors,https://www.youtube.com/results?search_query=robotics+AI+simulation+explained+students,17,Video,25,Embodied AI\nV011,AI in Medicine: Useful but Careful,AI in Medicine,Beginner,"Explains medical AI with a focus on decision support, data bias, privacy, and validation.",medicine;healthcare;privacy;bias,https://www.youtube.com/results?search_query=AI+in+medicine+explained+for+students,13,Video,28,AI for science\nV012,Open-Source AI Models and Fine-Tuning,Open Source AI,Intermediate,"Introduces open-source models, fine-tuning, and why reproducibility matters.",open+source;fine+tuning;models;reproducibility,https://www.youtube.com/results?search_query=open+source+AI+fine+tuning+explained,18,Tutorial,8,Accessible AI\n'
sessions_csv = 'session_id,title,topic,date,difficulty,teacher,description,capacity,available\nS001,Build a Mini Recommender in Python,Machine Learning,2026-07-12,Beginner,James,"Hands-on tutorial: use survey data and content tags to recommend videos.",30,Yes\nS002,How Neural Networks Learn,Deep Learning,2026-07-19,Beginner,James,"Visual demo of layers, weights, activation, loss, and gradient descent.",25,Yes\nS003,RAG Workshop: Give Your AI a Library,Retrieval-Augmented Generation,2026-07-26,Intermediate,James,"Build a tiny retrieval system over club notes and article summaries.",20,Yes\nS004,AI Safety Debate and Case Studies,AI Safety,2026-08-02,Beginner,Club Panel,"Discuss bias, hallucination, privacy, academic integrity, and responsible use.",35,Yes\nS005,Deploy a Streamlit AI App to GitHub,Deployment,2026-08-09,Beginner,James,"Turn a notebook prototype into a public portfolio project.",25,Yes\nS006,Reinforcement Learning Game Lab,Reinforcement Learning,2026-08-16,Intermediate,James,"Use a simple bandit game to show exploration, exploitation, and rewards.",20,Yes\nS007,Multimodal AI Show and Tell,Multimodal AI,2026-08-23,Beginner,Guest Mentor,"Explore AI that reads images, captions, audio, and text together.",30,Yes\nS008,AI for Science Lightning Talks,AI in Medicine,2026-08-30,Beginner,Club Members,"Student-led short talks about AI in health, biology, physics, and climate.",40,Yes\n'
users_csv = 'member_id,name,grade,preferred_topics,desired_level,math_comfort,coding_comfort,time_budget_minutes\nU001,Avery,9,"AI Agents;Generative AI;Prompting",Beginner,2,2,30\nU002,Brianna,10,"AI Safety;AI in Medicine;Machine Learning",Beginner,3,2,25\nU003,Carlos,11,"Deep Learning;Robotics;Deployment",Intermediate,4,4,45\nU004,Divya,12,"Retrieval-Augmented Generation;Open Source AI;Deep Learning",Intermediate,5,5,60\nU005,Ethan,9,"Reinforcement Learning;AI Agents;Robotics",Beginner,2,3,35\nU006,Fatima,10,"Multimodal AI;Generative AI;AI Safety",Beginner,3,3,30\n'
interactions_csv = 'member_id,content_id,watched,liked,rating,feedback_text\nU001,V001,1,1,5,"Agents feel useful for everyday school work."\nU001,V008,1,1,4,"Image generation was fun and easy to understand."\nU001,V006,1,0,2,"Too hardware focused for me."\nU001,V004,1,1,4,"Safety examples were helpful."\nU002,V004,1,1,5,"I liked the ethics and alignment part."\nU002,V011,1,1,5,"Medical examples made AI feel meaningful."\nU002,V010,1,0,2,"Robotics was less interesting."\nU002,V009,1,1,4,"Evaluation tips are useful for school projects."\nU003,V002,1,1,5,"Attention and neural nets are interesting."\nU003,V010,1,1,5,"I want to build robots."\nU003,V001,1,0,3,"Agents are okay but I want deeper math."\nU003,V006,1,1,4,"Edge AI sounds practical."\nU004,V007,1,1,5,"RAG connects AI to real evidence."\nU004,V012,1,1,5,"Open-source fine-tuning is exciting."\nU004,V008,1,0,3,"Less interested in images."\nU004,V002,1,1,4,"Good theory overview."\nU005,V003,1,1,5,"Rewards and games make sense."\nU005,V010,1,1,4,"Robotics is cool."\nU005,V011,1,0,2,"Medicine is not my main interest."\nU005,V001,1,1,4,"Agents are like game characters that plan."\nU006,V005,1,1,5,"Models that see and hear are amazing."\nU006,V008,1,1,4,"Diffusion models were creative."\nU006,V004,1,1,4,"Safety matters."\nU006,V006,1,0,2,"Edge devices felt too technical."\n'
threads_csv = 'thread_id,channel,author,topic,post,timestamp,upvotes\nT001,ai-agents,Avery,AI Agents,"Could an AI agent plan a full science fair project but still let the student do the thinking?",2026-07-01 16:20,12\nT002,ai-safety,Brianna,AI Safety,"I think every demo should include what could go wrong and how to test it.",2026-07-02 18:05,15\nT003,deep-learning,Carlos,Deep Learning,"The attention analogy helped me understand why transformers can handle long text.",2026-07-03 14:45,9\nT004,rag-and-tools,Divya,Retrieval-Augmented Generation,"RAG feels like giving the AI a school library card before it answers.",2026-07-04 11:30,18\nT005,robotics,Ethan,Robotics,"Can reinforcement learning train a robot in simulation before it tries the real world?",2026-07-04 19:10,7\nT006,multimodal,Fatima,Multimodal AI,"I want a session where we compare image captions from different models.",2026-07-05 10:00,10\n'

videos = pd.read_csv(StringIO(videos_csv))
sessions = pd.read_csv(StringIO(sessions_csv), parse_dates=["date"])
users = pd.read_csv(StringIO(users_csv))
interactions = pd.read_csv(StringIO(interactions_csv))
threads = pd.read_csv(StringIO(threads_csv))

display(videos.head())
display(users.head())
display(interactions.head())

## 2. Supervised machine learning recommender

Goal: predict whether a student will like a video. We use profile text, video metadata, and previous like labels. This is a lightweight but realistic supervised ML setup.

In [ ]:
def profile_text(row):
    return (
        f"preferred topics {row['preferred_topics']} desired level {row['desired_level']} "
        f"grade {row['grade']} math comfort {row['math_comfort']} coding comfort {row['coding_comfort']} "
        f"time budget {row['time_budget_minutes']}"
    )

def video_text(row):
    return f"{row['title']} {row['topic']} {row['level']} {row['summary']} {row['tags']} {row['frontier_theme']}"

train = interactions.merge(users, on="member_id").merge(videos, on="content_id")
train["pair_text"] = train.apply(
    lambda r: (
        f"Member: {r['preferred_topics']} {r['desired_level']} grade {r['grade']} "
        f"math {r['math_comfort']} coding {r['coding_comfort']}. "
        f"Content: {r['title']} {r['topic']} {r['level']} {r['summary']} {r['tags']}"
    ),
    axis=1,
)

ml_model = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
ml_model.fit(train["pair_text"], train["liked"])

print("Training rows:", len(train))
print("Labels:")
print(train["liked"].value_counts())

In [ ]:
def recommend_for_user(member_id, top_n=5, hide_watched=True):
    user = users.loc[users.member_id == member_id].iloc[0]
    items = videos.copy()
    if hide_watched:
        watched = set(interactions.loc[(interactions.member_id == member_id) & (interactions.watched == 1), "content_id"])
        items = items.loc[~items.content_id.isin(watched)].copy()

    pair_texts = [f"Member: {profile_text(user)}. Content: {video_text(row)}" for _, row in items.iterrows()]
    supervised_prob = ml_model.predict_proba(pair_texts)[:, 1]

    docs = [profile_text(user)] + [video_text(row) for _, row in items.iterrows()]
    tfidf = TfidfVectorizer(stop_words="english", min_df=1).fit_transform(docs)
    similarity = cosine_similarity(tfidf[0], tfidf[1:]).ravel()

    preferred = [p.strip().lower() for p in str(user.preferred_topics).split(";")]
    topic_match = np.array([1.0 if t.lower() in preferred else 0.0 for t in items.topic])
    freshness = 1.0 / (1.0 + items.freshness_days.to_numpy() / 30.0)

    items["supervised_like_probability"] = supervised_prob
    items["content_similarity"] = similarity
    items["topic_match"] = topic_match
    items["freshness_score"] = freshness
    items["recommendation_score"] = (
        0.50 * supervised_prob +
        0.25 * similarity +
        0.15 * topic_match +
        0.10 * freshness
    )
    return items.sort_values("recommendation_score", ascending=False)

recs = recommend_for_user("U001", top_n=5)
display(recs[["content_id", "title", "topic", "level", "recommendation_score"]].head(5))

## 3. Deep neural network concept demo

A production app might use a larger model or a safe retrieval-augmented generation system. In this free notebook, we demonstrate the neural-network idea with a small MLP classifier that learns from the same member-content examples.

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english", min_df=1, max_features=300)
X = vectorizer.fit_transform(train["pair_text"])
y = train["liked"].astype(int).to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

nn_model = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    activation="relu",
    solver="adam",
    max_iter=1000,
    random_state=42,
)
nn_model.fit(X_train, y_train)
y_pred = nn_model.predict(X_test)

print("Neural-network demo accuracy:", round(accuracy_score(y_test, y_pred), 3))
print(classification_report(y_test, y_pred, zero_division=0))

## 4. High-school explanation engine

This function turns complex AI topics into a simple structure: big idea, analogy, why it matters, and a journal club activity.

In [ ]:
EXPLAINER = {
        "AI Agents": {
            "big_idea": "An AI agent is a system that can set a goal, choose steps, use tools, and check progress.",
            "analogy": "Think of it like a careful lab partner that makes a plan, uses tools, and checks the answer.",
            "why": "Agents move AI from answering one question toward helping with multi-step projects.",
            "activity": "Give an agent a safe task and ask it to show its plan before acting.",
        },
        "Deep Learning": {
            "big_idea": "Deep learning uses layers of artificial neurons to turn raw data into useful patterns.",
            "analogy": "It is like a team of students passing notes, with each layer adding one more clue.",
            "why": "It powers language models, image generators, speech tools, and many AI systems.",
            "activity": "Draw a three-layer network that predicts whether a student might like a video.",
        },
        "Reinforcement Learning": {
            "big_idea": "Reinforcement learning teaches an AI through rewards and penalties.",
            "analogy": "It is like learning a video game: try a move, see the score, then improve.",
            "why": "It helps with games, robots, recommendations, and feedback-based systems.",
            "activity": "Design a reward rule for a robot that should reach a goal without bumping into walls.",
        },
        "Retrieval-Augmented Generation": {
            "big_idea": "RAG lets an AI look up relevant documents before it answers.",
            "analogy": "Instead of guessing from memory, the AI gets a library card and checks notes first.",
            "why": "It can connect answers to trusted club resources and reduce unsupported claims.",
            "activity": "Give the AI three paragraphs and ask it to answer only from those paragraphs.",
        },
    }

    def explain_topic(topic, learner_context=""):
        card = EXPLAINER.get(topic, {
            "big_idea": f"{topic} can be understood by asking what data it uses, what pattern it learns, and how people check it.",
            "analogy": "It is like learning a new school activity: learn rules, try examples, improve with feedback.",
            "why": "The key is to connect inputs, process, and outputs.",
            "activity": "Write one example input and one example output for this topic.",
        })
        text = (
            f"Big idea: {card['big_idea']}

"
            f"High-school analogy: {card['analogy']}

"
            f"Why it matters: {card['why']}

"
            f"Try it in journal club: {card['activity']}"
        )
        if learner_context:
            text += f"

Personal connection: relate this to {learner_context}."
        return text

    print(explain_topic("Reinforcement Learning", "robotics and games"))

## 5. Reinforcement learning feedback loop

This lightweight bandit-style loop updates topic weights. A real app would store these weights in a database per user.

In [ ]:
rl_weights = {}

def update_feedback(member_id, topic, reward, learning_rate=0.20):
    rl_weights.setdefault(member_id, {})
    old = rl_weights[member_id].get(topic, 0.0)
    rl_weights[member_id][topic] = round(old + learning_rate * reward, 3)
    return rl_weights[member_id][topic]

def recommend_with_rl(member_id):
    recs = recommend_for_user(member_id, hide_watched=False).copy()
    weights = rl_weights.get(member_id, {})
    recs["rl_score"] = recs.topic.map(lambda t: np.tanh(weights.get(t, 0.0)))
    recs["final_score"] = recs.recommendation_score + 0.05 * recs.rl_score
    return recs.sort_values("final_score", ascending=False)

print("Before feedback")
display(recommend_with_rl("U001")[["title", "topic", "final_score"]].head(5))

update_feedback("U001", "AI Agents", reward=1.0)
update_feedback("U001", "Deployment", reward=-1.0)

print("After feedback")
display(recommend_with_rl("U001")[["title", "topic", "rl_score", "final_score"]].head(5))

## 6. Visualize topic coverage

In [ ]:
topic_counts = videos["topic"].value_counts().sort_values(ascending=True)
ax = topic_counts.plot(kind="barh", figsize=(8, 5), title="Sample content by AI topic")
ax.set_xlabel("Number of videos")
ax.set_ylabel("Topic")
plt.show()

## 7. Export data for Streamlit

Run this cell in Colab if you want to save the CSV files generated in the notebook.

In [ ]:
from pathlib import Path

out_dir = Path("ai_journal_club_data")
out_dir.mkdir(exist_ok=True)
videos.to_csv(out_dir / "videos.csv", index=False)
sessions.to_csv(out_dir / "sessions.csv", index=False)
users.to_csv(out_dir / "user_profiles.csv", index=False)
interactions.to_csv(out_dir / "interactions.csv", index=False)
threads.to_csv(out_dir / "discussions.csv", index=False)

print("Saved CSV files to", out_dir.resolve())

## 8. Streamlit packaging plan

The downloadable ZIP that comes with this notebook already contains a complete Streamlit app. To run it locally:

```bash
pip install -r requirements.txt
streamlit run app.py
```

For GitHub, upload the whole project folder, then deploy `app.py` on Streamlit Community Cloud.